In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive

### Load Model

In [2]:
model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MODEL.pkl')
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/teamInfo.py
Updated 12 teams with confirmed lineups


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, edge_threshold=0.20, stake=10, 
                     variance_inflation=1.1, distribution_type='t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV%', ascending=False).reset_index(drop=True)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head()

Processing single bets with single model...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,MODEL PROB,EDGE,EV%,KELLY_FRACTION,KELLY_DOLLARS,CONFIDENCE INTERVAL,INTERVAL WIDTH,SIGMA,SIGMA FLAG,EXPECTED ROI,SIMULATION_METHOD
0,Goga Bitadze,BetMGM,player_points,4.5,105,Over,11.21,1,0.911,0.089,0.488,0.911,0.423,8.67,0.826,2.62,"(0.3, 22.1)",21.73,5.54,Med,0.87,Monte Carlo
1,Steven Adams,BetMGM,player_points,4.5,-125,Over,7.77,0,0.754,0.246,0.556,0.754,0.199,3.58,0.448,2.00,"(0.0, 18.7)",18.72,5.59,Med,0.36,Monte Carlo
2,Marvin Bagley III,BetMGM,player_points,5.5,-140,Over,8.80,0,0.742,0.258,0.583,0.742,0.159,2.73,0.382,1.79,"(0.0, 20.5)",20.49,5.96,Med,0.27,Monte Carlo
3,Tari Eason,BetMGM,player_points,10.5,-118,Over,12.74,0,0.673,0.327,0.541,0.673,0.132,2.44,0.288,2.12,"(1.2, 24.2)",23.00,5.87,Med,0.24,Monte Carlo
4,Tari Eason,BetOnline.ag,player_points,10.5,-123,Over,12.74,0,0.673,0.327,0.552,0.673,0.122,2.21,0.271,2.03,"(1.2, 24.2)",23.00,5.87,Med,0.22,Monte Carlo


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
underdogPairs = underdogPairs[
    underdogPairs[['sigma_flag1', 'sigma_flag2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Marvin Bagley III,Steven Adams,7.5,4.5,8.8,7.77,over,over,0.597,0.754,0.4055,0.019,0.176,0.098,0.22,0.108,0,"(0.0, 20.5)","(0.0, 18.7)",20.49,18.72,5.96,5.59,Med,Med,Monte Carlo
1,Marvin Bagley III,Tari Eason,7.5,10.5,8.8,12.74,over,over,0.597,0.673,0.3618,0.019,0.095,0.057,0.09,0.043,0,"(0.0, 20.5)","(1.2, 24.2)",20.49,23.00,5.96,5.87,Med,Med,Monte Carlo


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
pairsPrizepicks = pairsPrizepicks[
    pairsPrizepicks[['sigma_flag1', 'sigma_flag2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Goga Bitadze,Steven Adams,5.0,4.5,11.21,7.77,over,over,0.896,0.754,0.6087,0.318,0.176,0.247,0.83,0.413,0,"(0.3, 22.1)","(0.0, 18.7)",21.73,18.72,5.54,5.59,Med,Med,Monte Carlo
1,Goga Bitadze,Tari Eason,5.0,10.5,11.21,12.74,over,over,0.896,0.673,0.5431,0.318,0.095,0.207,0.63,0.315,0,"(0.3, 22.1)","(1.2, 24.2)",21.73,23.00,5.54,5.87,Med,Med,Monte Carlo


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg
underdogTrios = threeLeg[
    threeLeg[['sigma_flag1', 'sigma_flag2', 'sigma_flag3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method
0,Marvin Bagley III,Tari Eason,Steven Adams,7.5,10.5,4.5,8.8,12.74,7.77,over,over,over,0.597,0.673,0.754,0.2457,0.019,0.095,0.176,0.097,0.47,0.095,0,"(0.0, 20.5)","(1.2, 24.2)","(0.0, 18.7)",20.49,23.0,18.72,5.96,5.87,5.59,Med,Med,Med,Monte Carlo


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg  
triosPrizepicks = threeLeg[
    threeLeg[['sigma_flag1', 'sigma_flag2', 'sigma_flag3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method
0,Goga Bitadze,Tari Eason,Steven Adams,5.0,10.5,4.5,11.21,12.74,7.77,over,over,over,0.869,0.653,0.717,0.3294,0.291,0.075,0.139,0.168,0.98,0.195,0,"(0.3, 22.1)","(1.2, 24.2)","(0.0, 18.7)",21.73,23.0,18.72,5.54,5.87,5.59,Med,Med,Med,Monte Carlo


In [10]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'LAST {window}'] = hits
        results[f'HIT RATE % LAST {window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
149,PrizePicks,player_points,Ryan Rollins,Over,13.5,-137,2025-11-01,2025-11-01T21:13:22Z
151,PrizePicks,player_points,Rudy Gobert,Over,11.0,-137,2025-11-01,2025-11-01T21:21:42Z
153,PrizePicks,player_points,Ryan Kalkbrenner,Over,9.5,-137,2025-11-01,2025-11-01T21:21:42Z
155,PrizePicks,player_points,LaMelo Ball,Over,25.5,-137,2025-11-01,2025-11-01T21:21:42Z
157,PrizePicks,player_points,Julius Randle,Over,24.5,-137,2025-11-01,2025-11-01T21:21:42Z
...,...,...,...,...,...,...,...,...
2376,PrizePicks,player_blocks_steals,Bilal Coulibaly,Over,1.5,-137,2025-11-01,2025-11-01T21:22:48Z
2378,PrizePicks,player_blocks_steals,Alperen Sengun,Over,1.5,-137,2025-11-02,2025-11-01T21:22:56Z
2380,PrizePicks,player_blocks_steals,Josh Minott,Over,1.5,-137,2025-11-02,2025-11-01T21:22:56Z
2382,PrizePicks,player_blocks_steals,Kevin Durant,Over,1.5,-137,2025-11-02,2025-11-01T21:22:56Z


In [11]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 67 records for player_points to player_points.csv
Saved 48 records for player_rebounds to player_rebounds.csv
Saved 29 records for player_assists to player_assists.csv
Saved 9 records for player_threes to player_threes.csv
Saved 7 records for player_blocks to player_blocks.csv
Saved 13 records for player_steals to player_steals.csv
Saved 35 records for player_field_goals to player_field_goals.csv
Saved 31 records for player_frees_made to player_frees_made.csv
Saved 13 records for player_frees_attempts to player_frees_attempts.csv
Saved 83 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 79 records for player_points_rebounds to player_points_rebounds.csv
Saved 76 records for player_points_assists to player_points_assists.csv
Saved 48 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 20 records for player_turnovers to player_turnovers.csv
Saved 9 records for player_blocks_steals to player_blocks_steals.csv

All category f

In [12]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 66 records for player_points to player_points.csv
Saved 20 records for player_rebounds to player_rebounds.csv
Saved 11 records for player_assists to player_assists.csv
Saved 10 records for player_threes to player_threes.csv
Saved 1 records for player_steals to player_steals.csv
Saved 3 records for player_frees_made to player_frees_made.csv
Saved 76 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 32 records for player_points_rebounds to player_points_rebounds.csv
Saved 26 records for player_points_assists to player_points_assists.csv
Saved 15 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 6 records for player_turnovers to player_turnovers.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
